@poleside 2025/11/4
read data from WW

In [1]:
import socket
import pandas as pd
import geopandas as gpd
import numpy as np
import os

### merge WW and RGI

In [2]:
# check rgi
import geopandas as gpd
rgi = r'C:\ML4GM\data\RGI7\RGI2000-v7.0-G-13_central_asia.shp'
gdf = gpd.read_file(rgi)
print(gdf.head())
print(len(gdf))
# print(gdf.columns)
# save as csv
output_csv_path = r'C:\ML4GM\data\RGI7\rgi_13.csv'
#    - index=False 表示在CSV中不保存 DataFrame 的索引（行号）
gdf.drop(columns='geometry').to_csv(output_csv_path, index=False)

                    rgi_id o1region o2region        glims_id  anlys_id  \
0  RGI2000-v7.0-G-13-00001       13    13-01  G067426E38743N    804440   
1  RGI2000-v7.0-G-13-00002       13    13-01  G067480E38714N    804446   
2  RGI2000-v7.0-G-13-00003       13    13-01  G067485E38713N    804448   
3  RGI2000-v7.0-G-13-00004       13    13-01  G067489E38714N    804451   
4  RGI2000-v7.0-G-13-00005       13    13-01  G067492E38714N    804453   

   subm_id             src_date     cenlon     cenlat  utm_zone  ...  \
0      752  2002-07-10T00:00:00  67.425881  38.743313        42  ...   
1      752  2002-07-10T00:00:00  67.479616  38.714583        42  ...   
2      752  2002-07-10T00:00:00  67.484971  38.713429        42  ...   
3      752  2002-07-10T00:00:00  67.489409  38.714494        42  ...   
4      752  2002-07-10T00:00:00  67.491937  38.713707        42  ...   

      zmin_m     zmax_m     zmed_m    zmean_m  slope_deg  aspect_deg  \
0  3693.8557  3783.9656  3727.2417  3728.6082  34.

In [3]:
# check rgi6
rgi = r'C:\ML4GM\data\RGI6\15_rgi60_SouthAsiaEast.shp'
gdf = gpd.read_file(rgi)
print(gdf.head())
print(len(gdf))
# print(gdf.columns)
# save as csv
output_csv_path = r'C:\ML4GM\data\RGI6\rgi_15.csv'
#    - index=False 表示在CSV中不保存 DataFrame 的索引（行号）
gdf.drop(columns='geometry').to_csv(output_csv_path, index=False)

            RGIId         GLIMSId   BgnDate   EndDate      CenLon     CenLat  \
0  RGI60-15.00001  G102044E29941N  19990920  -9999999  102.044042  29.941000   
1  RGI60-15.00002  G102042E29987N  19990920  -9999999  102.042346  29.987019   
2  RGI60-15.00003  G102041E29997N  19990920  -9999999  102.041130  29.997311   
3  RGI60-15.00004  G102050E29962N  19990920  -9999999  102.050283  29.962297   
4  RGI60-15.00005  G102044E30025N  19990920  -9999999  102.043728  30.025101   

  O1Region O2Region   Area  Zmin  ...  Aspect  Lmax  Status  Connect  Form  \
0       15        3  0.438  4996  ...     251   850       0        0     0   
1       15        3  0.644  4947  ...     244  1021       0        0     0   
2       15        3  0.225  5019  ...     274   812       0        0     0   
3       15        3  0.985  4622  ...      52  2318       0        0     0   
4       15        3  0.465  4733  ...      20   913       0        0     0   

   TermType  Surging  Linkages  Name  \
0         

In [4]:
# merge RGI6 and WW data
merged_data = []

print("开始处理冰川数据...")

for i in range(13, 16):
    print(f"\n--- 正在处理区域 {i} ---")
    try:
        # 2. 加载 RGI 属性数据 和 WW 动态数据
        df_rgi = pd.read_csv(f'C:/ML4GM/data/RGI6/rgi_{i}.csv')
        df_ww = pd.read_csv(f'C:/ML4GM/data/glacierMass/dh_{i}_rgi60_pergla_rates.csv')

        print(f"区域 {i}: 加载了 {len(df_rgi)} 条 RGI 记录, {len(df_ww)} 条 WW 记录)")

        # 3. 创建用于连接的 'join_key'
        # 假设 rgi_id 和 rgiid 的最后五位是共同的 ID
        df_rgi['join_key'] = df_rgi['RGIId'].str[-5:]
        df_ww['join_key'] = df_ww['rgiid'].str[-5:]
        print(f"rgi冰川数{df_rgi['join_key'].nunique()}")
        print(f"ww冰川数{df_ww['join_key'].nunique()}")

        # 4. 执行左连接 (Left Merge)
        # 保留 df_ww (左表) 中的所有行 并将 df_rgi (右表) 中匹配 'join_key' 的属性添加过来
        merged_df = pd.merge(df_ww, df_rgi, on='join_key', how='left')
        
        # 注意: 如果 df_rgi 和 df_ww 有其他同名列 (除了'join_key')
        # pandas 会自动添加 '_x' (来自 df_ww) 和 '_y' (来自 df_rgi) 后缀。

        print(f"区域 {i}: 连接完成。")

        # 5. 将处理好的区域数据添加到列表中
        merged_data.append(merged_df)

    except FileNotFoundError as e:
        print(f"错误: 找不到文件 {e.filename}。请检查路径。")
    except Exception as e:
        print(f"处理区域 {i} 时发生错误: {e}")

# 6. 循环结束后，将所有区域的数据合并成一个 DataFrame
if merged_data:
    ww_rgi = pd.concat(merged_data, ignore_index=True)

    print("\n--- 所有区域数据合并完成 ---")
    print(f"总行数: {len(ww_rgi)}")
    print("merged data head:")
    print(ww_rgi.head())
    
    # 7. (可选) 将最终的 DataFrame 保存为 CSV 文件
    output_path = 'C:/ML4GM/data/ww_rgi.csv'
    ww_rgi.to_csv(output_path, index=False)
    print(f"saved as: {output_path}")

else:
    print("\n没有处理任何数据 请检查文件路径或循环范围")
    ww_rgi = pd.DataFrame() # 创建一个空的 DataFrame

开始处理冰川数据...

--- 正在处理区域 13 ---
区域 13: 加载了 54429 条 RGI 记录, 2286018 条 WW 记录)
rgi冰川数54429
ww冰川数54429
区域 13: 连接完成。

--- 正在处理区域 14 ---
区域 14: 加载了 27988 条 RGI 记录, 1175496 条 WW 记录)
rgi冰川数27988
ww冰川数27988
区域 14: 连接完成。

--- 正在处理区域 15 ---
区域 15: 加载了 13119 条 RGI 记录, 550998 条 WW 记录)
rgi冰川数13119
ww冰川数13119
区域 15: 连接完成。

--- 所有区域数据合并完成 ---
总行数: 4012512
merged data head:
            rgiid                 period      area    dhdt  err_dhdt   dvoldt  \
0  RGI60-13.00001  2000-01-01_2001-01-01  432000.0  0.1924    3.4158  83118.0   
1  RGI60-13.00001  2000-01-01_2002-01-01  432000.0  0.2226    1.6643  96143.0   
2  RGI60-13.00001  2000-01-01_2004-01-01  432000.0  0.2067    0.8416  89301.0   
3  RGI60-13.00001  2000-01-01_2005-01-01  432000.0  0.1841    0.6705  79542.0   
4  RGI60-13.00001  2000-01-01_2010-01-01  432000.0  0.0928    0.3335  40091.0   

   err_dvoldt      dmdt  err_dmdt  dmdtda  ...  Slope  Aspect  Lmax  Status  \
0   1475680.0  0.000071  0.001254  0.1635  ...   22.0     312   683       0

In [ ]:
# select by lat lon
lat_min = 27
lat_max = 32
lon_min = 92
lon_max = 99

slc_ww_rgi = ww_rgi[ww_rgi['CenLat'].between(lat_min, lat_max)  
                    & ww_rgi['CenLon'].between(lon_min, lon_max)]
print(f"selected data count: {len(slc_ww_rgi)}")
slc_ww_rgi.to_csv('C:/ML4GM/data/slc_ww_rgi.csv', index=False)

selected data count: 379344


In [6]:
print(slc_ww_rgi.head())

                rgiid                 period      area    dhdt  err_dhdt  \
25452  RGI60-13.00607  2000-01-01_2001-01-01  439000.0 -1.3969    8.3013   
25453  RGI60-13.00607  2000-01-01_2002-01-01  439000.0 -1.3898    4.0775   
25454  RGI60-13.00607  2000-01-01_2004-01-01  439000.0 -1.3727    1.9709   
25455  RGI60-13.00607  2000-01-01_2005-01-01  439000.0 -1.3630    1.5483   
25456  RGI60-13.00607  2000-01-01_2010-01-01  439000.0 -1.3120    0.7714   

         dvoldt  err_dvoldt      dmdt  err_dmdt  dmdtda  ...  Slope  Aspect  \
25452 -613236.0   3644670.0 -0.000521  0.003098 -1.1874  ...   29.7     329   
25453 -610133.0   1790843.0 -0.000519  0.001523 -1.1814  ...   29.7     329   
25454 -602599.0    866861.0 -0.000512  0.000738 -1.1668  ...   29.7     329   
25455 -598360.0    681739.0 -0.000509  0.000581 -1.1586  ...   29.7     329   
25456 -575986.0    342378.0 -0.000490  0.000293 -1.1152  ...   29.7     329   

       Lmax  Status  Connect  Form TermType Surging Linkages  Name  

In [11]:
# count glaciers by rgi
total = 0
for i in range(13, 16):
    df_rgi = pd.read_csv(f'C:/ML4GM/data/RGI6/rgi_{i}.csv')

    lat_min = 27
    lat_max = 46
    lon_min = 67
    lon_max = 104
    slc_df_rgi = df_rgi[df_rgi['CenLat'].between(lat_min, lat_max)  
                        & df_rgi['CenLon'].between(lon_min, lon_max)]
    count = len(slc_df_rgi)
    print(f"rgi{i}: {count}")
    total = total + count
print(f"total glaciers: {total}")


rgi13: 54429
rgi14: 27988
rgi15: 13119
total glaciers: 95536


### process ERA5

In [1]:
# 必须先 import dask（且要在 xarray 之前），xarray 的 chunks= 才能用 dask
import dask
import dask.array
from dask.diagnostics import ProgressBar
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_origin
from scipy.interpolate import RegularGridInterpolator
from netCDF4 import Dataset
from tqdm import tqdm  # 进度条显示
import os

In [2]:
# interpolate with xarray + dask (keep attrs/encoding)

def bilinear_interpolation_era5(input_file, output_file, target_res=0.005,
                                time_chunk=1, lat_chunk=400, lon_chunk=1000):
    """使用 xarray+dask 对 ERA5 数据做空间双线性插值，保留时间维度。

    - 分辨率提高到 target_res（默认 0.005°），时间范围/变量不变；
    - 利用 chunk 分块计算，避免一次性创建 (time, lat, lon) 巨型数组；
    - 尽量保留原始 NetCDF 的变量/坐标/时间轴属性和 encoding。
    """

    # 懒加载打开，按时间维分块以触发 dask（若 dask 不可用则不分块，可能占内存）
    try:
        ds = xr.open_dataset(input_file, chunks={"valid_time": time_chunk})
    except ImportError as e:
        if "dask" in str(e).lower():
            import warnings
            warnings.warn("dask 不可用，将不分块打开文件，大数据集可能占较多内存。建议：先运行上方 import 单元（且 dask 在 xarray 之前），或重启 kernel 后重试。")
            ds = xr.open_dataset(input_file)
        else:
            raise

    # 原始坐标
    lat = ds["latitude"]
    lon = ds["longitude"]
    time = ds["valid_time"]

    # 计算新网格范围
    lat_min = float(lat.min())
    lat_max = float(lat.max())
    lon_min = float(lon.min())
    lon_max = float(lon.max())

    new_lats = np.arange(lat_min, lat_max + target_res, target_res)
    new_lons = np.arange(lon_min, lon_max + target_res, target_res)

    # 需要插值的变量
    vars_to_interp = ["t2m", "tp"]
    ds_sub = ds[vars_to_interp]

    # 利用 xarray 的 interp 在空间维做双线性插值
    ds_interp = ds_sub.interp(latitude=new_lats, longitude=new_lons)

    # 复制变量/坐标/全局属性
    for v in vars_to_interp:
        ds_interp[v].attrs = ds[v].attrs

    for coord in ["latitude", "longitude", "valid_time"]:
        if coord in ds.coords and coord in ds_interp.coords:
            ds_interp[coord].attrs = ds[coord].attrs

    ds_interp.attrs = ds.attrs

    # netCDF4 后端只支持下列 encoding 键，原文件可能含 szip/zstd/coordinates 等需过滤掉
    _nc4_enc_keys = {
        "chunksizes", "blosc_shuffle", "dtype", "complevel", "fletcher32",
        "compression", "significant_digits", "least_significant_digit", "zlib",
        "contiguous", "quantize_mode", "szip_coding", "shuffle", "_FillValue",
        "endian", "szip_pixels_per_block",
    }

    def _filter_nc4_encoding(enc):
        return {k: v for k, v in enc.items() if k in _nc4_enc_keys}

    # 构造 encoding：只保留 netCDF4 支持的键，再设置 dtype/压缩/chunk
    encoding = {}
    n_lat = ds_interp.sizes["latitude"]
    n_lon = ds_interp.sizes["longitude"]

    for v in vars_to_interp:
        base_enc = _filter_nc4_encoding(ds[v].encoding)
        base_enc["dtype"] = "float64"
        base_enc.setdefault("zlib", True)
        base_enc.setdefault("complevel", 4)
        base_enc["chunksizes"] = (
            time_chunk,
            min(lat_chunk, n_lat),
            min(lon_chunk, n_lon),
        )
        encoding[v] = base_enc

    # 坐标变量编码只继承 netCDF4 支持的键
    for coord in ["latitude", "longitude", "valid_time"]:
        if coord in ds.coords and coord in ds_interp.coords:
            encoding[coord] = _filter_nc4_encoding(ds[coord].encoding)

    # 写出 NetCDF，dask 会按 chunk 分块计算（ProgressBar 显示进度）
    with ProgressBar():
        ds_interp.to_netcdf(output_file, encoding=encoding, compute=True)

    ds.close()
    ds_interp.close()

    print(f"插值完成，已保存至 {output_file}")


# 示例用法
input_nc = r"C:\ML4GM\data\ERA5\ALL_t2mTp_ERA.nc" 
interpolated_nc = input_nc.replace('.nc', '_interpolated.nc')
bilinear_interpolation_era5(input_nc, interpolated_nc)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_20700\1456831038.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(input_file, chunks={"valid_time": time_chunk})


[########################################] | 100% Completed | 49m 50s
插值完成，已保存至 C:\ML4GM\data\ERA5\ALL_t2mTp_ERA_interpolated.nc


In [13]:
import geopandas as gpd
import os

regions_to_count = [13, 14, 15]
base_shp_dir = r"C:\ML4GM\proc_data\03_RGI_A2"

region_counts = {}
total_count = 0

for region in regions_to_count:
    shp_path = os.path.join(base_shp_dir, f"{region}_A2.shp")
    if os.path.exists(shp_path):
        gdf = gpd.read_file(shp_path)
        num_elements = len(gdf)
        region_counts[region] = num_elements
        total_count += num_elements
        print(f"区域 {region} 的要素数量: {num_elements}")
    else:
        region_counts[region] = 0
        print(f"区域 {region} 的shp文件未找到 ({shp_path})")

print(f"总要素数量: {total_count}")

区域 13 的要素数量: 4197
区域 14 的要素数量: 2469
区域 15 的要素数量: 1435
总要素数量: 8101


In [6]:
import xarray as xr
import geopandas as gpd
import rioxarray
import pandas as pd
import os
from collections import defaultdict

# --- 参数设置：13、14、15 三区 ---
nc_path = r"C:\ML4GM\data\ERA5/ALL_t2mTp_ERA_interpolated.nc"
variables = ["t2m", "tp"]
regions = [14, 15]
base_shp_dir = r"C:\ML4GM\proc_data\03_RGI_A2"
base_out_dir = r"C:\ML4GM\proc_data\01_glc_era"
grid_deg = 1.0  # 空间批处理网格步长（度），同网格内多边形共读一次 NC，越大 I/O 越少、单块内存越大

def calculate_time_series_average_optimized(shp_path, nc_path, variables, output_csv, grid_deg=1.0):
    try:
        print("正在读取数据...")
        shp_gdf = gpd.read_file(shp_path)
        ds = xr.open_dataset(nc_path, mask_and_scale=True, chunks={"valid_time": 24})[variables].astype("float32")
        ds = ds.chunk({"valid_time": 24, "latitude": 500, "longitude": 500})
        if ds.rio.crs is None:
            ds.rio.write_crs("EPSG:4326", inplace=True)
        if shp_gdf.crs != ds.rio.crs:
            print(f"正在统一坐标系: {shp_gdf.crs} -> {ds.rio.crs}")
            shp_gdf = shp_gdf.to_crs(ds.rio.crs)
        y_dim, x_dim = ds.rio.y_dim, ds.rio.x_dim

    except Exception as e:
        print(f"数据准备阶段出错: {e}")
        return

    results_list = []
    total = len(shp_gdf)

    # 按空间网格分组：同一网格内多边形共读一次 NC，再在内存中逐要素 clip/mean
    groups = defaultdict(list)
    for index, row in shp_gdf.iterrows():
        feature_id = row.get('RGIId', f'ID_{index}')
        geom = row.geometry
        minx, miny, maxx, maxy = geom.bounds
        cell = (int(minx // grid_deg), int(miny // grid_deg))
        groups[cell].append((feature_id, geom))
    n_cells = len(groups)
    print(f"开始处理，共 {total} 个要素、{n_cells} 个空间批（网格 {grid_deg}°）...")

    done = 0
    for cell_idx, (cell, rows_in_cell) in enumerate(groups.items()):
        print(f"批 {cell_idx + 1}/{n_cells}（本批 {len(rows_in_cell)} 个要素）")
        minx = min(g.bounds[0] for _, g in rows_in_cell)
        miny = min(g.bounds[1] for _, g in rows_in_cell)
        maxx = max(g.bounds[2] for _, g in rows_in_cell)
        maxy = max(g.bounds[3] for _, g in rows_in_cell)
        try:
            tile = ds.rio.clip_box(minx, miny, maxx, maxy).compute()
        except Exception as e:
            print(f"  跳过该批 clip_box: {e}")
            continue
        for feat_idx, (feature_id, geom) in enumerate(rows_in_cell):
            done += 1
            if done % 10 == 0 or done == total or feat_idx == 0:
                print(f"  要素 {done}/{total} (ID: {feature_id})")
            try:
                clipped_ds = tile.rio.clip([geom], tile.rio.crs, drop=True, all_touched=True)
                time_series_mean = clipped_ds.mean(dim=[y_dim, x_dim])
                temp_df = time_series_mean.to_dataframe().reset_index()
                cols_to_keep = [c for c in temp_df.columns if c in variables or 'time' in c.lower()]
                temp_df = temp_df[cols_to_keep]
                temp_df['RGIId'] = feature_id
                results_list.append(temp_df)
            except Exception as e:
                print(f"  跳过要素 {feature_id}: {e}")
        del tile

    # 4. 合并并保存结果
    if results_list:
        print("正在整合数据...")
        final_results_df = pd.concat(results_list, ignore_index=True)

        # 统一时间列名
        # 寻找包含 'time' 字样的列并重命名为 'time'
        time_col = [c for c in final_results_df.columns if 'time' in c.lower()][0]
        final_results_df = final_results_df.rename(columns={time_col: 'time'})

        # 整理列顺序：[ID, time, var1, var2...]
        cols = ['RGIId', 'time'] + [v for v in variables if v in final_results_df.columns]
        final_results_df = final_results_df[cols]

        # 保存
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        final_results_df.to_csv(output_csv, index=False)
        print(f"任务完成！结果已存至: {output_csv}")
    else:
        print("没有提取到任何有效数据。")

for region in regions:
    shp_path = os.path.join(base_shp_dir, f"{region}_A2.shp")
    output_csv = os.path.join(base_out_dir, f"glcera_{region}.csv")
    if not os.path.exists(shp_path):
        print(f"跳过区域 {region}：未找到 {shp_path}")
        continue
    print(f"--- 区域 {region} ---")
    calculate_time_series_average_optimized(shp_path, nc_path, variables, output_csv, grid_deg=grid_deg)

--- 区域 14 ---
正在读取数据...
开始处理，共 2469 个要素、45 个空间批（网格 1.0°）...
批 1/45（本批 89 个要素）
  要素 1/2469 (ID: RGI60-14.00005)
  要素 10/2469 (ID: RGI60-14.04706)
  要素 20/2469 (ID: RGI60-14.04856)
  要素 30/2469 (ID: RGI60-14.04942)
  要素 40/2469 (ID: RGI60-14.05019)
  要素 50/2469 (ID: RGI60-14.05618)
  要素 60/2469 (ID: RGI60-14.07207)
  要素 70/2469 (ID: RGI60-14.10103)
  要素 80/2469 (ID: RGI60-14.19150)
批 2/45（本批 54 个要素）
  要素 90/2469 (ID: RGI60-14.00018)
  要素 100/2469 (ID: RGI60-14.04872)
  要素 110/2469 (ID: RGI60-14.19164)
  要素 120/2469 (ID: RGI60-14.19417)
  要素 130/2469 (ID: RGI60-14.20143)
  要素 140/2469 (ID: RGI60-14.20212)
批 3/45（本批 171 个要素）
  要素 144/2469 (ID: RGI60-14.00032)
  要素 150/2469 (ID: RGI60-14.00142)
  要素 160/2469 (ID: RGI60-14.00572)
  要素 170/2469 (ID: RGI60-14.00727)
  要素 180/2469 (ID: RGI60-14.00842)
  要素 190/2469 (ID: RGI60-14.01022)
  要素 200/2469 (ID: RGI60-14.01206)
  要素 210/2469 (ID: RGI60-14.01580)
  要素 220/2469 (ID: RGI60-14.01948)
  要素 230/2469 (ID: RGI60-14.02192)
  要素 240/2469 (ID: RG

In [ ]:
# 转换数据格式
import pandas as pd

# 13、14、15 三区循环
import os
input_dir = r"C:\ML4GM\proc_data\01_glc_era"
output_dir = r"C:\ML4GM\proc_data\01_glc_era"
os.makedirs(output_dir, exist_ok=True)
regions = [13, 14, 15]

for region in regions:
    input_path = os.path.join(input_dir, f"glcera_{region}.csv")
    if not os.path.exists(input_path):
        print(f"跳过区域 {region}：未找到 {input_path}")
        continue
    print(f"转换区域 {region}...")
    df = pd.read_csv(input_path)

    df['time'] = pd.to_datetime(df['time'])
    df['Year'] = df['time'].dt.year
    df['Month'] = df['time'].dt.month
    df_t2m = df.pivot(index=['RGIId', 'Year'], columns='Month', values='t2m')
    df_t2m.columns = [f"{m}_t2m" for m in df_t2m.columns]
    df_tp = df.pivot(index=['RGIId', 'Year'], columns='Month', values='tp')
    df_tp.columns = [f"{m}_tp" for m in df_tp.columns]
    df_transformed = pd.concat([df_t2m, df_tp], axis=1).reset_index()
    output_path = os.path.join(output_dir, f"glcera_{region}_transformed.csv")
    df_transformed.to_csv(output_path, index=False)
    print(f"  已保存: {output_path}")

    print(df_transformed.head())

转换区域 13...
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_13_transformed.csv
转换区域 14...
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_14_transformed.csv
转换区域 15...
  已保存: C:\ML4GM\proc_data\01_glc_era\glcera_15_transformed.csv


In [10]:
import pandas as pd

def process_glacier_annual_data(input_path, output_path):
    print(f"正在读取文件: {input_path} ...")
    df = pd.read_csv(input_path)
    
    # 定义内部函数：检查是否为年度数据 (YYYY-01-01 到 YYYY+1-01-01)
    def extract_year_if_annual(period_str):
        try:
            # 预期格式: 2000-01-01_2001-01-01
            start_date, end_date = str(period_str).split('_')
            s_year = int(start_date[:4])
            e_year = int(end_date[:4])
            
            # 验证：1月1日开始，1月1日结束，且正好跨越1年
            if start_date.endswith('-01-01') and end_date.endswith('-01-01') and (e_year == s_year + 1):
                return s_year
            return None
        except:
            # 格式不匹配或缺失值返回 None
            return None

    # 1. 提取年份
    df['year'] = df['period'].apply(extract_year_if_annual)
    
    # 2. 筛选出符合条件的年度数据
    df_annual = df[df['year'].notnull()].copy()
    
    # 3. [新增筛选] 仅筛选出属性 Area >= 2 的数据
    # 注意：如果你的 CSV 中列名是小写的 area，请将 Area 改为 area
    if 'Area' in df_annual.columns:
        df_annual = df_annual[df_annual['Area'] >= 2].copy()
    else:
        print("警告：未在文件中找到 'Area' 列，跳过面积筛选。")
    
    # 4. 格式转换
    df_annual['year'] = df_annual['year'].astype(int)
    
    # 5. 整理列结构：移除原 period，将 year 置于原 period 的位置
    cols = list(df_annual.columns)
    if 'period' in cols:
        period_idx = cols.index('period')
        cols.pop(cols.index('year'))  # 移除末尾的 year
        cols.insert(period_idx, 'year') # 插入到原 period 的位置
        df_annual = df_annual[cols].drop(columns=['period'])
    
    # 6. 保存结果
    df_annual.to_csv(output_path, index=False)
    
    print("-" * 30)
    print(f"处理成功！")
    print(f"原始数据总行数: {len(df)}")
    print(f"符合年度且 Area >= 2 的行数: {len(df_annual)}")
    print(f"结果已保存至: {output_path}")

# --- 使用方法 ---
# 使用 r"" 原始字符串防止 Windows 路径转义错误
input_file = r"C:\ML4GM\data\ww_rgi.csv"
output_file = r"C:\ML4GM\proc_data\02_merge\ww_rgi_formed.csv"

process_glacier_annual_data(input_file, output_file)

正在读取文件: C:\ML4GM\data\ww_rgi.csv ...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_21592\3527817517.py:5: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


------------------------------
处理成功！
原始数据总行数: 4012512
符合年度且 Area >= 2 的行数: 162020
结果已保存至: C:\ML4GM\proc_data\02_merge\ww_rgi_formed.csv


In [14]:
import pandas as pd
import os

def merge_and_filter_glcera(input_paths, output_path):
    # 只读取存在的文件
    existing = [p for p in input_paths if os.path.exists(p)]
    if not existing:
        raise FileNotFoundError(f"未找到任何输入文件: {input_paths}")
    dfs = [pd.read_csv(p) for p in existing]
    df_merged = pd.concat(dfs, ignore_index=True)
    df_filtered = df_merged[(df_merged['Year'] >= 2000) & (df_merged['Year'] <= 2019)].copy()
    df_filtered = df_filtered.sort_values(by=['RGIId', 'Year'])
    df_filtered.to_csv(output_path, index=False)

# --- 13、14、15 三区合并 ---
base_dir = r"C:\ML4GM\proc_data\01_glc_era"
input_paths = [
    os.path.join(base_dir, "glcera_13_transformed.csv"),
    os.path.join(base_dir, "glcera_14_transformed.csv"),
    os.path.join(base_dir, "glcera_15_transformed.csv"),
]
output_file = r"C:\ML4GM\proc_data\02_merge\glcera_yeared_merged.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
merge_and_filter_glcera(input_paths, output_file)

In [16]:
import pandas as pd

def merge_glacier_and_climate_data(ww_path, glc_path, output_path):
    print("正在加载数据...")
    # 读取冰川年度数据 (WW)
    df_ww = pd.read_csv(ww_path)
    # 读取气候数据 (GLCERA)
    df_glc = pd.read_csv(glc_path)
    
    # 1. 统一列名和数据类型
    # 将 GLCERA 中的 'Year' 统一为小写的 'year' 以匹配 WW 数据
    if 'Year' in df_glc.columns:
        df_glc = df_glc.rename(columns={'Year': 'year'})
    
    # 确保用于合并的列类型一致 (均为整数)
    df_ww['year'] = df_ww['year'].astype(int)
    df_glc['year'] = df_glc['year'].astype(int)
    
    # 2. 合并数据 (内连接 Inner Join)
    # 基于 RGIId 和 year 两个键进行合并
    print("正在进行数据合并...")
    df_merged = pd.merge(df_ww, df_glc, on=['RGIId', 'year'], how='inner')
    
    # 3. 保存结果
    df_merged.to_csv(output_path, index=False)
    
    print("-" * 30)
    print("处理成功！")
    print(f"冰川数据行数: {len(df_ww)}")
    print(f"气候数据行数: {len(df_glc)}")
    print(f"合并后的有效数据行数: {len(df_merged)}")
    print(f"结果已保存至: {output_path}")

# --- 使用方法 ---
# 请根据你的实际路径修改文件名
ww_file = r"C:\ML4GM\proc_data\02_merge\ww_rgi_formed.csv"
glc_file = r"C:\ML4GM\proc_data\02_merge\glcera_yeared_merged.csv"
output_file = r"C:\ML4GM\proc_data\02_merge\merged_data.csv"

merge_glacier_and_climate_data(ww_file, glc_file, output_file)

正在加载数据...
正在进行数据合并...
------------------------------
处理成功！
冰川数据行数: 162020
气候数据行数: 162020
合并后的有效数据行数: 162020
结果已保存至: C:\ML4GM\proc_data\02_merge\merged_data.csv


In [17]:
# 删除不需要的列
df = pd.read_csv(output_file)
columns_to_drop = ['area', 'err_dhdt', 'dvoldt', 'err_dvoldt', 'dmdt', 'err_dmdt', 'dmdtda', 
                   'err_dmdtda', 'perc_area_meas', 'perc_area_res', 'valid_obs', 'valid_obs_py', 
                   'reg', 'join_key', 'RGIId', 'BgnDate', 'EndDate', 'O1Region', 'O2Region', 
                   'Status', 'Connect', 'Form', 'TermType', 'Surging', 'Linkages', 'Name']

df_cleaned = df.drop(columns=columns_to_drop)
# 根据output_file路径，改个名字保存
cleaned_output_file = output_file.replace('.csv', '_cleaned.csv')
df_cleaned.to_csv(cleaned_output_file, index=False)
print(df_cleaned.head())

            rgiid  year    dhdt         GLIMSId   CenLon   CenLat   Area  \
0  RGI60-13.00062  2000  0.4247  G078112E35641N  78.1118  35.6407  2.222   
1  RGI60-13.00062  2001  0.4973  G078112E35641N  78.1118  35.6407  2.222   
2  RGI60-13.00062  2002  0.3004  G078112E35641N  78.1118  35.6407  2.222   
3  RGI60-13.00062  2003  0.1291  G078112E35641N  78.1118  35.6407  2.222   
4  RGI60-13.00062  2004  0.4655  G078112E35641N  78.1118  35.6407  2.222   

   Zmin  Zmax  Zmed  ...      3_tp      4_tp      5_tp      6_tp      7_tp  \
0  5417  6011  5839  ...  0.000586  0.000365  0.000449  0.000739  0.002704   
1  5417  6011  5839  ...  0.000269  0.000451  0.000659  0.002461  0.001985   
2  5417  6011  5839  ...  0.000421  0.000794  0.000548  0.000775  0.001459   
3  5417  6011  5839  ...  0.000706  0.000823  0.000717  0.000891  0.000666   
4  5417  6011  5839  ...  0.000467  0.000711  0.000561  0.001879  0.001524   

       8_tp      9_tp     10_tp     11_tp     12_tp  
0  0.002098  0.00043